In [1]:
import json
from pathlib import Path

with open("experiments/bm25_metrics.json") as f:
    bm25_metrics=json.load(f)
bm25_metrics

{'ndcg@10': 0.683408827231414, 'map': 0.6447050264550264}

In [6]:
from beir import util 
from beir.datasets.data_loader import GenericDataLoader

data_path="data/raw/msmarco"
corpus,queries,qrels=GenericDataLoader(data_path).load(split="dev")

100%|█████████████████████████████| 8841823/8841823 [00:16<00:00, 541453.04it/s]


In [7]:
print(type(corpus))
print(type(queries))
print(type(qrels))


<class 'dict'>
<class 'dict'>
<class 'dict'>


In [10]:
len(corpus)


8841823

In [15]:
# Relevant docs from qrels
relevant_doc_ids = set()
for qid, docs in qrels.items():
    for d in docs:
        relevant_doc_ids.add(d)

filtered_corpus = {d: corpus[d] for d in relevant_doc_ids}

# Sample queries (same size)
import random
random.seed(42)
sampled_qids = random.sample(list(queries.keys()), 300)
sampled_queries = {qid: queries[qid] for qid in sampled_qids}

len(filtered_corpus), len(sampled_queries)


(7433, 300)

In [16]:
from rank_bm25 import BM25Okapi

doc_ids = list(filtered_corpus.keys())
docs_text = [filtered_corpus[d]["text"] for d in doc_ids]
tokenized_docs = [d.lower().split() for d in docs_text]
bm25 = BM25Okapi(tokenized_docs)

def build_candidates(query, qid, k_pos=2, k_neg=8):
    scores = bm25.get_scores(query.lower().split())
    ranked = sorted(zip(doc_ids, scores), key=lambda x: x[1], reverse=True)

    positives = list(qrels.get(qid, {}).keys())[:k_pos]
    negatives = []
    for doc_id, _ in ranked:
        if doc_id not in positives:
            negatives.append(doc_id)
        if len(negatives) >= k_neg:
            break
    return positives, negatives


In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1,2))

all_texts = list(sampled_queries.values()) + [filtered_corpus[d]["text"] for d in doc_ids]
tfidf = vectorizer.fit_transform(all_texts)

q_vecs = tfidf[:len(sampled_queries)]
d_vecs = tfidf[len(sampled_queries):]


In [18]:
def overlap_feats(q, d):
    qset, dset = set(q.lower().split()), set(d.lower().split())
    return {
        "common_terms": len(qset & dset),
        "overlap_ratio": len(qset & dset) / (len(qset) + 1e-6)
    }


In [20]:
import pandas as pd

rows = []

qid_list = list(sampled_qids)

for qi, qid in enumerate(qid_list):
    query = sampled_queries[qid]
    pos, neg = build_candidates(query, qid)

    for doc_id in pos + neg:
        di = doc_ids.index(doc_id)

        tfidf_score = cosine_similarity(q_vecs[qi], d_vecs[di])[0][0]
        bm25_scores = bm25.get_scores(query.lower().split())
        bm25_score = bm25_scores[di]
        overlap = overlap_feats(query, filtered_corpus[doc_id]["text"])

        label = 1 if doc_id in pos else 0

        rows.append({
            "query_id": qid,
            "doc_id": doc_id,
            "tfidf": tfidf_score,
            "bm25": bm25_score,
            "common_terms": overlap["common_terms"],
            "overlap_ratio": overlap["overlap_ratio"],
            "label": label
        })

features_df = pd.DataFrame(rows)
features_df.head()


,query_id,doc_id,tfidf,bm25,common_terms,overlap_ratio,label
0,1033703,7763416,0.170245,17.398290,2,0.50,1
1,1033703,7763417,0.243869,17.860031,2,0.50,1
2,1033703,4521109,0.201760,15.901173,3,0.75,0
3,1033703,7667700,0.000000,11.077825,2,0.50,0
4,1033703,7480492,0.000000,11.012561,2,0.50,0


In [21]:
Path("data/processed").mkdir(exist_ok=True)
features_df.to_csv("data/processed/features_ltr.csv", index=False)

features_df.shape


(2717, 7)

In [22]:
features_df["label"].value_counts()


label
0    2400
1     317
Name: count, dtype: int64

In [23]:
features_df.describe()


,tfidf,bm25,common_terms,overlap_ratio,label
count,2717.000000,2717.000000,2717.000000,2717.000000,2717.000000
mean,0.084207,13.704057,2.884063,0.501387,0.116673
std,0.091251,5.863820,1.345981,0.155034,0.321089
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.031698,9.951473,2.000000,0.400000,0.000000
50%,0.057409,12.826974,3.000000,0.500000,0.000000
75%,0.104074,16.290767,4.000000,0.600000,0.000000
max,0.718574,51.579194,9.000000,1.000000,1.000000
